In [ ]:
''This code represents the process of engineering features for the exogenous variables gathered for this experiment. 
It generates lags and rolling means for both raw series and residuals, depending on the input.

In [1]:
# Feature Engineering 

In [1]:
# === Cell 0: Libraries ===

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import normaltest
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
%pip install xlsxwriter
import xlsxwriter

print("✅ Libraries loaded.")


Note: you may need to restart the kernel to use updated packages.
✅ Libraries loaded.


In [6]:
# === Cell 1: EDA for All Variables ===

import os
import pandas as pd
import numpy as np
from scipy.stats import normaltest
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

# Load dataset
filename = 'b100_structured_res.csv'  
assert os.path.exists(filename), "File not found. Please check the filename."
df = pd.read_csv(filename)

# Ensure 'date' is datetime and set as index
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])
df.set_index('date', inplace=True)
df = df.asfreq('D')

# Forward fill to avoid NaNs for rolling and STL
df_ffill = df.ffill()

# Excel writer
eda_excel = filename.replace('.csv', '_EDA_ALL.xlsx')
writer = pd.ExcelWriter(eda_excel, engine='xlsxwriter')

# EDA Summary Collection
eda_summary_list = []

for col in df.columns:
    try:
        series = df_ffill[col]
        original_series = df[col]

        stats = series.describe(percentiles=[.25, .5, .75])
        miss = original_series.isna().sum()
        norm_p = normaltest(series.dropna()).pvalue if len(series.dropna()) > 7 else np.nan

        try:
            stl = STL(series.dropna(), period=365).fit()
            trend = stl.trend
            slope = (trend.iloc[-1] - trend.iloc[0]) / len(trend)
        except:
            slope = np.nan

        adf_stat, adf_pvalue, *_ = adfuller(series.dropna())

        print(f"\n=== EDA for {col} ===")
        print(stats)
        print(f"Missing: {miss}")
        print(f"Normality p-value: {norm_p:.4f}" if not np.isnan(norm_p) else "Normality p-value: n/a")
        print(f"Trend slope: {slope:.6f}" if not np.isnan(slope) else "Trend slope: n/a")
        print(f"ADF Statistic: {adf_stat:.4f}, p-value: {adf_pvalue:.4f}, Stationary: {'Yes' if adf_pvalue <= 0.05 else 'No'}")

        eda_summary_list.append([
            col, miss, norm_p, slope, adf_stat, adf_pvalue,
            'Yes' if adf_pvalue <= 0.05 else 'No'
        ])

        # Save per-column stats to Excel
        pd.DataFrame(stats).to_excel(writer, sheet_name=f"{col}_stats")
    except Exception as e:
        print(f"Skipping column {col} due to error: {e}")

# Save summary to Excel
eda_summary_df = pd.DataFrame(eda_summary_list, columns=[
    'Variable', 'Missing', 'Normality p', 'Trend Slope',
    'ADF Statistic', 'ADF p-value', 'Stationary'
])
eda_summary_df.to_excel(writer, sheet_name='Summary', index=False)
writer.close()

print(f"\n📊 EDA results exported to: {eda_excel}")



=== EDA for b100_structured_res ===
count    4710.000000
mean        0.477427
std         1.636243
min        -6.545864
25%        -0.122873
50%         0.201309
75%         1.933568
max         2.198395
Name: b100_structured_res, dtype: float64
Missing: 1790
Normality p-value: 0.0000
Trend slope: 0.000418
ADF Statistic: -2.0645, p-value: 0.2590, Stationary: No

📊 EDA results exported to: b100_structured_res_EDA_ALL.xlsx


In [7]:
# === CELL 2: Self-contained Lag & Rolling Mean Feature Generator ===
import pandas as pd
import os

# 🔧 CONFIGURE HERE: Load your Excel or CSV data directly
input_file = "b100_structured_res.csv"  # ⬅️ Change this as needed
file_base = os.path.splitext(os.path.basename(input_file))[0]

df = pd.read_csv(input_file, index_col=0)
df.index = pd.to_datetime(df.index)
df = df.sort_index()

# Define lag and rolling windows
lag_days = {
    'lag_1_day': 1,
    'lag_2_days': 2,
    'lag_3_days': 3,
    'lag_1_week': 7,
    'lag_1_month': 30,
    'lag_3_month': 90,
    'lag_6_month': 180,
    'lag_1_year': 365,
    'lag_2_year': 730
}

rolling_windows = {
    'rolling_mean_1_week': 7,
    'rolling_mean_1_month': 30,
    'rolling_mean_3_month': 90,
    'rolling_mean_6_month': 180,
    'rolling_mean_1_year': 365,
    'rolling_mean_2_year': 730
}

df_features = df.copy()

# Generate lag features
for col in df.columns:
    for name, shift_days in lag_days.items():
        new_col = f"{col}_{name}"
        df_features[new_col] = df[col].shift(shift_days)

# Generate rolling mean features
for col in df.columns:
    for name, window in rolling_windows.items():
        new_col = f"{col}_{name}"
        df_features[new_col] = df[col].rolling(window=window, min_periods=1).mean()

# Export to CSV with correct dynamic filename
output_file = f"{file_base}_engineered_features.csv"
df_features.to_csv(output_file)

print(f"✅ Features saved to: {output_file}")


✅ Features saved to: b100_structured_res_engineered_features.csv


In [4]:
# === CELL 3: Plotting Features from Engineered File ===
import pandas as pd
import matplotlib.pyplot as plt

# 🔧 CONFIGURE HERE
variable_to_be_seen = "full_consumption_passage_res"                # ← Set your variable name here
input_base_name = "full_consumption_passage"      # ← Must match input file used in Cell 2

# Load the correct engineered CSV file
engineered_file = f"{input_base_name}_engineered_features.csv"
df_features = pd.read_csv(engineered_file, index_col=0)
df_features.index = pd.to_datetime(df_features.index)

# Plot settings
plt.figure(figsize=(15, 7))
plt.plot(df_features.index, df_features[variable_to_be_seen], label=variable_to_be_seen)

# Lags
# plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_lag_1_day"], label='Lag 1 Day', linestyle='--')
# plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_lag_2_days"], label='Lag 2 Days', linestyle='--')
# plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_lag_1_week"], label='Lag 1 Week', linestyle='--')
# plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_lag_1_year"], label='Lag 1 Year', linestyle='--')
plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_lag_2_year"], label='Lag 2 Years', linestyle='--')

# Rolling Means
# plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_rolling_mean_1_month"], label='Rolling Mean 1 Month', linestyle=':')
plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_rolling_mean_2_year"], label='Rolling Mean 2 Years', linestyle=':')

plt.title(f"Feature Plot – '{variable_to_be_seen}'")
plt.xlabel("Date")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'full_consumption_passage_engineered_features.csv'

In [ ]:
#Auxiliary codes:


In [ ]:
# === Toggle below lines ON/OFF to choose which features to show ===
# Lags
#plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_lag_1_day"], label='Lag 1 Day', linestyle='--')
#plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_lag_2_days"], label='Lag 2 Days', linestyle='--')
#plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_lag_1_week"], label='Lag 1 Week', linestyle='--')
#plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_lag_1_month"], label='Lag 1 Month', linestyle='--')
#plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_lag_1_year"], label='Lag 1 Year', linestyle='--')
plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_lag_2_year"], label='Lag 2 Years', linestyle='--')

# Rolling Means
#plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_rolling_mean_1_week"], label='Rolling Mean 1 Week', linestyle=':')
#plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_rolling_mean_1_month"], label='Rolling Mean 1 Month', linestyle=':')
#plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_rolling_mean_4_month"], label='Rolling Mean 4 Month', linestyle=':')
#plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_rolling_mean_6_month"], label='Rolling Mean 6 Month', linestyle=':')
#plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_rolling_mean_1_year"], label='Rolling Mean 1 Year', linestyle=':')
plt.plot(df_features.index, df_features[f"{variable_to_be_seen}_rolling_mean_2_year"], label='Rolling Mean 2 Years', linestyle=':')